In [9]:
import pandas as pd
import numpy as np

In [10]:
# Load the raw dataset
# Path is relative to the notebooks folder
try:
    df = pd.read_csv('../data/raw_it_assets.csv')
    print("✅ Data loaded successfully.")
except FileNotFoundError:
    print("❌ File not found. Please check the data folder.")

✅ Data loaded successfully.


In [11]:
# Initial data inspection
df.head()

,Asset_ID,Asset_Type,Assigned_To,Department,Acquisition_Date,Warranty_Expiry_Date,Status,Maintenance_Cost,License_Expiry_Date,Vendor_Name,Location,Compliance_Status,OS_Version,Repair_Count
0,ASSET_0000,Desktop,NaN,Other,2020-05-10,NaN,Decommissioned,NaN,NaN,Cisco,Other,Non-Compliant,Win 10,99
1,ASSET_0001,Laptop,User_187,Finance,2018/11/27,2021-02-27,Stock,180.0,NaN,Other,Other,At Risk,Win 10,3
2,ASSET_0002,Other,User_376,IT,30/08/2024,2026-03-09,In Use,71.0,NaN,Cisco,Cloud,Compliant,Linux-Generic,2
3,ASSET_0003,Monitor,User_799,Other,2023-08-10,2025-11-27,In Use,736.0,NaN,Other,Taichung,Non-Compliant,NaN,3
4,ASSET_0004,Monitor,User_940,Other,2019/10/09,2020-10-28,In Use,650.0,NaN,Lenovo,Hsinchu,At Risk,NaN,0


In [12]:
# Check data types and overall structure
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 14 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Asset_ID              500 non-null    object 
 1   Asset_Type            500 non-null    object 
 2   Assigned_To           416 non-null    object 
 3   Department            500 non-null    object 
 4   Acquisition_Date      500 non-null    object 
 5   Warranty_Expiry_Date  450 non-null    object 
 6   Status                500 non-null    object 
 7   Maintenance_Cost      458 non-null    float64
 8   License_Expiry_Date   109 non-null    object 
 9   Vendor_Name           500 non-null    object 
 10  Location              500 non-null    object 
 11  Compliance_Status     500 non-null    object 
 12  OS_Version            287 non-null    object 
 13  Repair_Count          500 non-null    int64  
dtypes: float64(1), int64(1), object(12)
memory usage: 54.8+ KB


In [13]:
# Check for missing values explicitly
df.isnull().sum()

Asset_ID                  0
Asset_Type                0
Assigned_To              84
Department                0
Acquisition_Date          0
Warranty_Expiry_Date     50
Status                    0
Maintenance_Cost         42
License_Expiry_Date     391
Vendor_Name               0
Location                  0
Compliance_Status         0
OS_Version              213
Repair_Count              0
dtype: int64

☞ Data Diagnostics: Insights from Missing Values

After running df.isnull().sum(), the results reveal specific operational patterns within the IT infrastructure rather than random data loss. This highlights the importance of Domain-Specific Imputation over simple data deletion.

1. Physical Inventory Status (Assigned_To)

Observation: There are 84 missing entries in the Assigned_To column.

Context: In a real-world IT environment, a missing assignee typically indicates the asset is physically located in the warehouse (buffer stock).

Strategy: I will fill these missing values with "In Stock" to maintain the integrity of the inventory analysis.

2. Asset Category Logic (License_Expiry_Date & OS_Version)

Observation: Significant missing values are found in these columns (391 and 213, respectively).

Context: This is logically consistent with the asset types. Peripherals like Monitors do not have Operating Systems, and only Software assets require license tracking.

Strategy: These will be handled using category-aware labels (e.g., "Not Applicable") to prevent bias in the machine learning model.

3. Maintenance History (Maintenance_Cost)

Observation: 42 records have missing maintenance costs.

Context: This suggests these assets have not undergone any repairs since acquisition.

Strategy: These will be imputed with 0 to accurately reflect the health and ROI of these "healthy" assets.

4. Data Recovery: Estimated Warranty (Warranty_Expiry_Date)

Observation: 50 records are missing warranty expiration dates.

Strategy: I implemented a recovery logic by estimating the expiry as 3 years (1095 days) from the Acquisition Date.

Conclusion: By choosing Logical Imputation instead of dropping rows, I preserve the full 500-record dataset. This allows the predictive model to learn the characteristics of "In-Stock" and "Zero-Maintenance" assets, which are critical for accurate lifecycle forecasting.

In [15]:
# Fill basic categorical and numerical nulls

# Fill Assigned_To: Empty values indicate the asset is physically in the warehouse.
df['Assigned_To'] = df['Assigned_To'].fillna('In-Stock')

# Fill Maintenance_Cost: Empty values suggest no repair costs incurred yet.
df['Maintenance_Cost'] = df['Maintenance_Cost'].fillna(0)

# Fill OS_Version: Non-computing devices (like monitors) don't have an OS.
df['OS_Version'] = df['OS_Version'].fillna('Not Applicable')